# Your first pysewer benchmark

This notebook walks through one complete pysewer run, step by step, using the
small dataset that ships with the repository — no extra downloads needed.

**Who is this for?** People new to pysewer (and maybe new to Python). Every
step explains *what* happens and *why*. If you want the fast, scriptable
version of the same run, look at `benchmarks/scripts/run_scenarios.py`.

**What pysewer does:** given three inputs — a terrain model (DEM), roads and
buildings — it designs a sewer network: where pipes go, how wide they are,
how deep they lie, and where pumps or lifting stations are needed.


## 1. Setup

We import pysewer and point it at the test dataset. The paths are relative
to the repository root, so we change the working directory first.


In [ ]:
import os
from pathlib import Path

# move to the repository root (the folder that contains 'pysewer' and 'tests')
while not Path('pysewer').is_dir():
    os.chdir('..')

import pysewer
print('pysewer imported from:', pysewer.__file__)

## 2. Configuration — *before* building the model

pysewer reads its parameters from a central configuration
(`pysewer/config/settings.yaml`). You can override any of them.

**Important rule:** apply your configuration *before* creating the
`ModelDomain` — some preprocessing options (`clustering`,
`connect_buildings`) are used while the model is being built and cannot be
changed afterwards.

Here we lower `tmax` (the maximum allowed trench depth) to 3 m to see how
the design reacts to a stricter constraint. Try 8.0 later and compare!


In [ ]:
pysewer.set_custom_config(custom_settings_dict={
    'optimization': {'tmax': 3.0}
})
print('tmax is now:', pysewer.get_config().optimization.tmax, 'm')

## 3. Build the model domain

The `ModelDomain` loads the DEM, roads and buildings, connects the
buildings to the road network and prepares the graph that all later steps
work on. This is the slowest step (expect ~a minute).


In [ ]:
model_domain = pysewer.ModelDomain(
    dem='tests/test_data/dem.tif',
    roads='tests/test_data/roads_clipped.shp',
    buildings='tests/test_data/buildings_clipped.shp',
)

# tell pysewer where the wastewater treatment plant (the 'sink') is
sink = (691350, 2553250)
model_domain.add_sink(sink)

## 4. Route the network

`generate_connection_graph` evaluates every road segment (can gravity flow
work here, or would it need a pump?). Segments that would need a pump get a
high routing cost (`pump_penalty`), so the router avoids them when it can.
`rsph_tree` then connects every building to the sink as cheaply as possible.


In [ ]:
connection_graph = model_domain.generate_connection_graph()
layout = pysewer.rsph_tree(connection_graph, [sink])

## 5. Size the pipes

`estimate_peakflow` computes how much wastewater flows through each pipe
(based on the connected inhabitants). `calculate_hydraulic_parameters` then
chooses pipe diameters, computes trench depths, and places pumping and
lifting stations where gravity is not enough.


In [ ]:
sewer = pysewer.estimate_peakflow(layout)
G = pysewer.calculate_hydraulic_parameters(sewer, sinks=[sink])

## 6. Look at the results

The result is a network (a `networkx` graph). `get_edge_gdf` turns the
pipes into a GeoDataFrame — a table you can filter, plot and export.

Key columns:
- `diameter` — chosen pipe diameter in metres
- `pressurized` — `True` where a pump pushes the water (the design flag)
- `peak_flow` — design flow in m³/s
- `hydraulic_violations` — honest flags where constraints could not be met


In [ ]:
from pysewer.helper import get_edge_gdf

pipes = get_edge_gdf(G, detailed=True)

print('pipes:', len(pipes))
print('total length [m]:', round(pipes.geometry.length.sum()))
print('diameters:', pipes['diameter'].value_counts().to_dict())
print('pressurized pipes:', int(pipes['pressurized'].sum()))
n_pump = sum(1 for _, d in G.nodes(data=True) if d.get('pumping_station'))
n_lift = sum(1 for _, d in G.nodes(data=True) if d.get('lifting_station'))
print('pumping stations:', n_pump, '| lifting stations:', n_lift)

## 7. Plot the network

Plotting needs matplotlib (already included in the standard conda
environment; on a light install add it with `pip install 'pysewer[plot]'`).


In [ ]:
info = pysewer.get_sewer_info(G)
fig, ax = pysewer.plot_model_domain(
    model_domain, plot_sewer=True, sewer_graph=G, info_table=info
)

## 8. Export for GIS

The export writes a GeoPackage you can open in QGIS. Float values are
rounded automatically (see `export.round_decimals` in the settings).


In [ ]:
out = Path('work/benchmarks/results')
out.mkdir(parents=True, exist_ok=True)
pysewer.export_sewer_network(pipes, str(out / 'notebook_run.gpkg'))

## 9. Where to go next

- Re-run with `tmax: 8.0` in step 2 and compare the number of stations —
  that is exactly what the benchmark scenarios automate.
- Run the scripted benchmarks: `python benchmarks/scripts/run_scenarios.py`
  (results land in `work/benchmarks/results/summary.md`).
- Add your own scenario YAML under `benchmarks/scenarios/` — see
  `benchmarks/README.md`.
